# ClimX U-Net — headless GPU run

Launched from WSL via `kaggle kernels push` (`make kaggle-push`). No browser needed: poll with `make kaggle-status`, pull results with `make kaggle-pull`.

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)

In [ ]:
# Attached via competition_sources; auto-detect the mount.
ncs = sorted(pathlib.Path("/kaggle/input").rglob("NorESM2-MM_r1i1p1f1_with_forcing_subsampled_16_*.nc"))
print(f"found {len(ncs)} lite files")
parents = {p.parent for p in ncs}
assert len(parents) == 1, f"lite files spread across dirs: {parents}"
os.environ["CLIMX_LITE_DIR"] = str(parents.pop())
os.environ["CLIMX_PREPROCESS_DIR"] = "/kaggle/working/data/preprocessing_data_lite"
print("CLIMX_LITE_DIR =", os.environ["CLIMX_LITE_DIR"])

In [ ]:
import climx_project
from climx_project import data, preprocess
from climx_project.paths import ensure_dirs
ensure_dirs()
ds = data.open_datasets()
preprocess.run_preprocess_train(ds)
preprocess.run_preprocess_test(ds)
print("preprocess done")

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "train.py", "experiment=unet"])
print("train exit code:", r.returncode)

In [ ]:
# Collect artifacts into /kaggle/working/outputs for `kaggle kernels output`.
import shutil, pathlib
out = pathlib.Path("/kaggle/working/outputs"); out.mkdir(exist_ok=True)
scores = pathlib.Path("results/scores.md")
if scores.exists():
    shutil.copy(scores, out / "scores.md")
tb = pathlib.Path("results/runs/tensorboard")
if tb.exists() and any(tb.rglob("events.out.tfevents.*")):
    shutil.make_archive(str(out / "tb-logs"), "zip", root_dir=tb)
for p in pathlib.Path("results/submissions").glob("submission_unet*.parquet"):
    shutil.copy(p, out / p.name)
print("outputs:", [f.name for f in sorted(out.iterdir())])